<a href="https://colab.research.google.com/github/dohee-jin/data-mining-final-project/blob/main/notebooks/kbo_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 골든글러브 후보 예측

- 선수 개인 기록을 활용해 2026년 골든글러브 후보 가능성이 높은 선수를 예측한다.
- 타자와 투수는 기록 컬럼이 다르므로 분리해서 분석한다.

---

## 2. 골든글로브 후보 예측 데이터 사용 전략

데이터는 크게 세 종류로 나눈다.

| 구분 | 출처 | 직접 정리 여부 | 사용 목적 |
|---|---|---:|---|
| 과거 선수 기록 | Kaggle KBO Player Dataset 1982-2025 | X | 골든글러브 예측 학습 데이터 |
| 2026 현재 선수 기록 | KBO 공식 홈페이지 기록실 | O | 2026 골든글러브 후보 예측 대상 |
| 골든글러브 수상자 목록 | KBO 공식 골든글러브 수상 현황 | O | `is_goldenglove` 라벨 생성 |

---

## 3. 골든글러브 후보 예측용 데이터

### 3.1 Kaggle에서 가져올 데이터

Kaggle의 `KBO Player Dataset (1982-2025)`를 과거 학습 데이터로 사용한다.

사용할 데이터는 다음과 같다.

```text
kbo_batting_stats_by_season_1982-2025.csv
kbo_pitching_stats_by_season_1982-2025.csv
```

Kaggle 데이터는 1982년부터 2025년까지의 KBO 정규시즌 선수 타격 및 투수 기록을 포함한다. 실제 분석에서는 너무 오래된 데이터까지 모두 쓰기보다, 최근 야구 환경과 비슷한 기간만 필터링하여 사용한다.

추천 사용 기간:

```text
2015~2025
```

### 3.2 직접 정리해야 하는 2026 선수 기록

2026년은 아직 시즌이 진행 중이므로 Kaggle 과거 데이터에 포함되어 있지 않다. 따라서 KBO 공식 홈페이지에서 현재 기록을 복사해 CSV로 직접 정리하고 깃허브에 csv로 raw 파일을 업로드했다.

```text
data/raw/kbo_hitter_data_2026.csv
data/raw/kbo_pitcher_data_2026.csv
```

2026 선수 기록 파일에는 반드시 `year` 컬럼을 추가한다.   
타자 기록에는 골든글러브 포지션별 예측을 위해 `position` 칼럼을 추가한다.   
일부 선수는 시즌 중 여러 포지션에 출전하므로, 본 분석에서는 데이터 단순화와 모델 적용의 일관성을 위해 선수별 대표 포지션을 하나만 부여하였다. 대표 포지션은 KBO 기록실의 선수 구분 또는 주요 출전 포지션을 기준으로 정리하였다.

```text
year = 2026
position = C, 1B, 2B, 3B, SS, OF, DH
```

예시:

| year | player | team | avg | hr | rbi | ops |
|---:|---|---|---:|---:|---:|---:|
| 2026 | 선수명 | LG | 0.315 | 12 | 45 | 0.890 |

### kagglehub로 kbo 1982-2025 데이터셋 불러오기

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("netsong/kbo-player-dataset-by-regular-season-1982-2025")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'kbo-player-dataset-by-regular-season-1982-2025' dataset.
Path to dataset files: /kaggle/input/kbo-player-dataset-by-regular-season-1982-2025


In [9]:
import os
import pandas as pd

path = kagglehub.dataset_download("netsong/kbo-player-dataset-by-regular-season-1982-2025")

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

Using Colab cache for faster access to the 'kbo-player-dataset-by-regular-season-1982-2025' dataset.
/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_pitching_stats_career_totals_1982-2025.csv
/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_batting_stats_career_totals_1982-2025.csv
/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_batting_stats_by_season_1982-2025.csv
/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_pitching_stats_by_season_1982-2025.csv


In [15]:
hitter = pd.read_csv("/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_batting_stats_by_season_1982-2025.csv")
pitcher = pd.read_csv("/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_pitching_stats_by_season_1982-2025.csv")

hitter.head()
pitcher.head()

# 2015~2025년 데이터만 필터링
hitter_recent = hitter[hitter["Year"] > 2015]
pitcher_recent = pitcher[pitcher["Year"] > 2015]

hitter_recent.head()
pitcher_recent.head()

,Id,Name,Birthdate,Handedness,School,Draft,Year,Team,Age,Pos.,...,SO,ROE,BK,WP,ERA,RA9,rRA9,FIP,WHIP,WAR
2399,10043,곽정철,1986년 03월 14일,우투우타,송정동초-무등중-광주제일고,05 KIA 1차,2016,KIA,30,P,...,16,0.0,1,1,7.90,7.90,7.67,6.81,2.05,-0.17
2415,10045,김건한,1981년 03월 26일,우투우타,포항초-포철중-포철공고,01 SK 2차 1라운드 1순위,2016,삼성,35,P,...,7,2.0,0,0,3.72,4.66,4.66,4.94,1.24,0.27
2448,10050,박경태,1987년 09월 02일,좌투좌타,숭의초-동산중-동산고,06 KIA 2차 3라운드 21순위,2017,KIA,30,P,...,3,1.0,0,1,9.00,13.50,13.50,4.66,2.50,-0.12
2449,10050,박경태,1987년 09월 02일,좌투좌타,숭의초-동산중-동산고,06 KIA 2차 3라운드 21순위,2018,KIA,31,P,...,6,1.0,0,1,5.68,7.11,9.81,3.45,1.42,-0.02
2468,10054,손영민,1987년 04월 28일,우언우타,우암초-청주중-청주기공고,06 KIA 2차 1라운드 5순위,2017,KIA,30,P,...,9,0.0,0,0,10.80,10.80,10.56,6.13,2.18,-0.66


### Github에 업로드 한 2026 raw 데이터 불러오기


In [17]:
golden_glove = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_goldenglove_data_2025.csv")
hitter_2026 = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_hitter_data_2026.csv")
pitcher_2026 = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_pitcher_data_2026.csv")

golden_glove.head()
hitter_2026.head()
pitcher_2026.head()

,rank,name,team,ERA,G,W,L,SV,HLB,WPCT,...,NP,AVG,2B,3B,SAC,SF,IBB,WP,BK,year
0,1,후라도,삼성,2.61,12,3,1,0,0,0.750,...,1156,0.259,6,2,4,3,0,4,0,2026
1,2,올러,KIA,2.66,13,7,5,0,0,0.583,...,1211,0.182,5,2,5,1,0,3,0,2026
2,3,류현진,한화,2.84,12,8,2,0,0,0.800,...,1059,0.239,14,1,7,2,1,1,0,2026
3,4,최민석,두산,2.88,12,6,2,0,0,0.750,...,1098,0.215,13,0,3,2,0,3,1,2026
4,5,알칸타라,키움,3.12,12,6,4,0,0,0.600,...,1154,0.259,12,2,1,1,0,2,0,2026


## 데이터 전처리
### 1. Kaggle 데이터와 KBO 2026 데이터 칼럼명 맞추기

#### 기록용어 참고
01. 타자 기록
  - 2B: 2루타
  - 3B: 3루타
  - AB: 타수
  - AO: 뜬공
  - AVG: 타율
  - BB: 볼넷
  - BB/K: 볼넷/삼진
  - CS: 도루실패
  - E: 실책
  - G: 경기
  - GDP: 병살타
  - GO : 땅볼
  - GO/AO : 땅볼/뜬공
  - GPA : (1.8x출루율+장타율)/4
  - GW RBI : 결승타
  - H : 안타
  - HBP(HP) : 사구
  - HR : 홈런
  - IBB(IB) : 고의4구
  - ISOP : 순수장타율
  - MH : 멀티히트
  - OBP : 출루율
  - OPS : 출루율+장타율
  - P/PA : 투구수/타석
  - PA : 타석
  - PH-BA : 대타타율
  - R : 득점
  - RBI : 타점
  - RISP : 득점권타율
  - SAC(SH) : 희생번트
  - SB : 도루
  - SF : 희생플라이
  - SLG : 장타율
  - SO : 삼진
  - TB: 루타
  - XBH : 장타
  - XR : 추정득점

02. 투수기록
  - 2B : 2루타
  - 3B : 3루타
  - AO : 뜬공
  - AVG : 피안타율
  - BABIP : 인플레이타구타율
  - BB : 볼넷
  - BB/9 : 9이닝당 볼넷
  - BK : 보크
  - BSV : 블론세이브
  - CG : 완투
  - ER : 자책점
  - ERA : 평균자책점
  - G : 경기
  - GDP : 병살타
  - GF : 종료
  - GO : 땅볼
  - GO/AO : 땅볼/뜬공
  - GS : 선발
  - H : 피안타
  - HBP(HP) : 사구
  - HLD(HD) : 홀드
  - HR : 홈런
  - IBB(IB) : 고의4구
  - IP : 이닝
  - K/9 : 9이닝당 삼진
  - K/BB : 삼진/볼넷
  - L : 패
  - NP : 투구수
  - OBP : 피출루율
  - OPS : 피출루율+피장타율
  - P/G : 투구수/경기
  - P/IP : 투구수/이닝
  - QS : 퀄리티스타트
  - R : 실점
  - SAC(SH) : 희생번트
  - SF : 희생플라이
  - SHO : 완봉
  - SLG : 피장타율
  - SO : 삼진
  - SV(S) : 세이브
  - SVO : 세이브기회
  - TBF : 타자수
  - TS : 터프세이브
  - W : 승
  - Wgr : 구원승
  - Wgs : 선발승
  - WHIP : 이닝당 출루허용률
  - WP : 폭투
  - WPCT : 승률

####
   

In [19]:
# 각 데이터의 칼럼명 확인
print(hitter_recent.columns)
print(pitcher_recent.columns)
print(hitter_2026.columns)
print(pitcher_2026.columns)
print(golden_glove.columns)


Index(['Id', 'Name', 'Birthdate', 'Handedness', 'School', 'Draft', 'Year',
       'Team', 'Age', 'Pos.', 'G', 'oWAR', 'dWAR', 'PA', 'ePA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SB', 'CS', 'BB', 'HP', 'IB', 'SO',
       'GDP', 'SH', 'SF', 'AVG', 'OBP', 'SLG', 'OPS', 'R/ePA', 'wRC+', 'WAR'],
      dtype='object')
Index(['Id', 'Name', 'Birthdate', 'Handedness', 'School', 'Draft', 'Year',
       'Team', 'Age', 'Pos.', 'G', 'GS', 'GR', 'GF', 'CG', 'SHO', 'W', 'L',
       'S', 'HD', 'IP', 'ER', 'R', 'rRA', 'TBF', 'H', '2B', '3B', 'HR', 'BB',
       'HP', 'IB', 'SO', 'ROE', 'BK', 'WP', 'ERA', 'RA9', 'rRA9', 'FIP',
       'WHIP', 'WAR'],
      dtype='object')
Index(['rank', 'name', 'team', 'position', 'AVG', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'BB', 'IBB', 'HBP', 'SO',
       'GDP', 'SLG', 'OBP', 'OPS', 'MH', 'RISP', 'PH-BA', 'year'],
      dtype='object')
Index(['rank', 'name', 'team', 'ERA', 'G', 'W', 'L', 'SV', 'HLB', 'WPCT', 'IP',


In [21]:
# 타자 데이터 컬럼명 정리(Kaggle 기준으로 정리)
hitter_2026 = hitter_2026.rename(columns={
    "name": "Name",
    "team": "Team",
    "year": "Year",
    "position": "Pos",
    "HBP": "HP",
    "IBB": "IB",
    "SAC": "SH"
})

# 투수 데이터 컬럼명 정리(Kaggle 기준으로 정리)
pitcher_2026 = pitcher_2026.rename(columns={
    "name": "Name",
    "team": "Team",
    "year": "Year",
    "HLB": "HD",
    "SV": "S"
})


print(hitter_2026.columns)
print(pitcher_2026.columns)

Index(['rank', 'Name', 'Team', 'Pos', 'AVG', 'G', 'PA', 'AB', 'R', 'H', '2B',
       '3B', 'HR', 'TB', 'RBI', 'SH', 'SF', 'BB', 'IB', 'HP', 'SO', 'GDP',
       'SLG', 'OBP', 'OPS', 'MH', 'RISP', 'PH-BA', 'Year'],
      dtype='object')
Index(['rank', 'Name', 'Team', 'ERA', 'G', 'W', 'L', 'S', 'HD', 'WPCT', 'IP',
       'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'CG', 'SHO', 'QS',
       'BSV', 'TBF', 'NP', 'AVG', '2B', '3B', 'SAC', 'SF', 'IBB', 'WP', 'BK',
       'Year'],
      dtype='object')


### 2. 골든글러브 데이터 전처리
#### 2-1. 칼럼명 정리
year, P, C, 1B, 2B, 3B, SS, OF1, OF2, OF3, OF4, DH(기존) --> year, position, player_team(전처리 후)

#### 2-2. 선수명/팀명 분리
현재 데이터에는 `폰세 한화`로 되어있어서 분리가 필요하다.

#### 2-3. 2015-2025 학습 데이터에 `is_goldenglove` 라벨 추가

In [29]:
# 2-1. 칼럼 및 데이터 정리
position_cols = ["P", "C", "1B", "2B", "3B", "SS", "OF1", "OF2", "OF3", "OF4", "DH"]

golden = golden_glove.melt(
    id_vars = "year",
    value_vars = position_cols,
    var_name = "position",
    value_name = "player_name"
)

# 빈 값 제거
golden = golden.dropna(subset=["player_name"])

# OF1..4 -> OF로 통일
golden["position"] = golden["position"].replace({
    "OF1": "OF",
    "OF2": "OF",
    "OF3": "OF",
    "OF4": "OF"
})

# 선수명, 팀이름 분리
golden[["name", 'team']] = golden["player_name"].str.rsplit(" ", n = 1, expand = True)
golden = golden[["year", "position", "name", "team"]]

golden.head()

,year,position,name,team
0,2025,P,폰세,한화
1,2024,P,하트,NC
2,2023,P,페디,NC
3,2022,P,안우진,키움
4,2021,P,미란다,두산


In [ ]:
# kaggle 학습 데이터에 is_goldenglove 라벨 추가
